<a href="https://colab.research.google.com/github/Satyadeep-Dey/genai-lab/blob/main/13_Agent_using_MCP_using_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**What this notebook demonstrates**

This notebook demonstrates how MCP is used to expose RAG capabilities which can be called by an Agent .

MCP will have the following tool :
*   **answer_question** : answers questions using my knowledge base via RAG




In [ ]:
!pip install -q openai faiss-cpu mcp

In [ ]:
#Imports
import sys
from google.colab import drive
import faiss
import pickle
import json

In [ ]:
# import a re-use function to read text from file stored in Google Drive
drive.mount('/content/drive', force_remount=True)
UTIL_PATH = "/content/drive/MyDrive/Colab Notebooks"
sys.path.insert(0, UTIL_PATH)
from file_utils_colab import read_text_from_file

# Let's use an anonymized and abridged version of Tale of Two Cities for RAG knowledge base
KB_text = read_text_from_file(
    folder_path="Files/Knowledge-Base",
    file_name= "Anonymized by OpenAI_TOTC.txt"
)

print(f"The number of characters are : {len(KB_text)}")

number_of_words = len(KB_text.split())
# Divides a string into a list of substrings based on a specified separator (default is whitespace) and then counts length of list

print(f"Number of words is : {number_of_words}")


In [ ]:
# Chunking the document
def chunk_text(text, chunk_size=300, overlap=50):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

In [ ]:
def build_index(doc,model):
    chunks = chunk_text(doc)
    embeddings = model.encode(chunks)
    faiss.normalize_L2(embeddings)

    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)

    faiss.write_index(index, "faiss.index")
    pickle.dump(chunks, open("chunks.pkl", "wb"))

In [ ]:
#Create embeddings using Hugging Face embedding model

from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")
build_index(KB_text,model)

In [ ]:
# Let's combine and test

import faiss
import pickle
from sentence_transformers import SentenceTransformer
from openai import OpenAI
from google.colab import userdata

# Sign in to OpenAI using Secrets in Colab
openai_api_key = userdata.get('OPENAI_API_KEY')
openai = OpenAI(api_key=openai_api_key)

# Load artifacts (same as MCP server will do)
model = SentenceTransformer("all-MiniLM-L6-v2")
index = faiss.read_index("faiss.index")

with open("chunks.pkl", "rb") as f:
    chunks = pickle.load(f)


def get_answer_local(question, threshold=0.45, k=5):
    # 1. Embed query
    q_embedding = model.encode([question])
    faiss.normalize_L2(q_embedding)

    # 2. Retrieve
    D, I = index.search(q_embedding, k=k)

    # 3. Filter
    #FAISS does vector search
    #Pickle stores text
    #Index links them via position
    filtered_chunks = [
        chunks[idx] for score, idx in zip(D[0], I[0]) if score >= threshold
    ]

    # 4. Handle no results
    if not filtered_chunks:
        return {"answer": "DO-NOT-KNOW", "sources": []}

    # 5. Build context
    context = "\n".join(filtered_chunks)

    # 6. Generate answer (your existing function)
    prompt = f"""
    Answer using ONLY the context below.
    If not found, say DO-NOT-KNOW.

    Context:
    {context}

    Question:
    {question}
    """

    # answer = message_gpt(system_message, prompt)

    # def message_gpt(system_message,prompt):
    messages = [
        {"role": "system", "content": "You are an assistant"},
        {"role": "user", "content": prompt}
      ]

    completion = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages,
    )
    answer = completion.choices[0].message.content
    return {
        "answer": answer,
        "sources": filtered_chunks[:3]
    }

In [ ]:
print(get_answer_local("Who is Philippe Duval married to ?"))

In [ ]:
print(get_answer_local("Who is the villain in this story ?",0.35)) # need to lower threshold to get correct answer

# **Let's create MCP server .**

In [ ]:
%%writefile mcp_server.py

from mcp.server.fastmcp import FastMCP
import faiss
import pickle
from sentence_transformers import SentenceTransformer
from openai import OpenAI
import os

mcp = FastMCP("rag-server")

openai_api_key = os.environ.get("OPENAI_API_KEY")
client = OpenAI(api_key=openai_api_key)
# The code in mcp_server.py works only when OPENAI_API_KEY is present
# in the environment at process start.
# If you don’t set it (via os.environ),then client is created with None and the
# API call will fail — so the server isn’t # “magically” accessing the key,
# it’s just inheriting it when you set it correctly.

model = SentenceTransformer("all-MiniLM-L6-v2")
index = faiss.read_index("faiss.index")

with open("chunks.pkl", "rb") as f:
    chunks = pickle.load(f)

def get_answer_impl(question, threshold=0.45, k=5):
    q_embedding = model.encode([question])
    faiss.normalize_L2(q_embedding)

    D, I = index.search(q_embedding, k=k)

    filtered_chunks = [
        chunks[idx] for score, idx in zip(D[0], I[0]) if score >= threshold
    ]

    if not filtered_chunks:
        return {"answer": "DO-NOT-KNOW", "sources": []}

    context = "\n".join(filtered_chunks[:3])

    prompt = f"""
Answer using ONLY the context below.
If not found, say DO-NOT-KNOW.

Context:
{context}

Question:
{question}
"""

    completion = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": "You are an assistant"},
            {"role": "user", "content": prompt}
        ],
    )

    answer = completion.choices[0].message.content

    return {
        "answer": answer,
        "sources": filtered_chunks[:3]
    }

@mcp.tool()
def get_answer(question: str, threshold: float = 0.45, k: int = 5) -> dict:
    return get_answer_impl(question, threshold, k)

if __name__ == "__main__":
    mcp.run()

In [ ]:
# MCP server runs in a separate process, and it only sees environment variables
# that exist at the moment that process is started.
# Without setting OPENAI_API_KEY the server process has no key → OpenAI call fails.

#Any time you start a separate process (not just in Colab),
#it won’t see notebook variables or secrets unless you pass them via environment variables.

import os
os.environ["OPENAI_API_KEY"] = #put your key here


In [ ]:
#Run server as subprocess (important)
import subprocess
import time

proc = subprocess.Popen(
    ["python", "mcp_server.py"],
    stdin=subprocess.PIPE,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
    bufsize=1
)

# Give server time to start
time.sleep(2)

# Check if server crashed
if proc.poll() is not None:
    print("Server crashed!")
    print(proc.stderr.read())
else:
    print("Server started")

In [ ]:
# Initialize MCP session
import json

def send_rpc(method, params=None, id=1):
    msg = {
        "jsonrpc": "2.0",
        "id": id,
        "method": method,
        "params": params or {}
    }

    proc.stdin.write(json.dumps(msg) + "\n")
    proc.stdin.flush()

    # Read until valid JSON
    while True:
        line = proc.stdout.readline()

        if not line:
            err = proc.stderr.read()
            raise RuntimeError(f"Empty response. STDERR:\n{err}")

        line = line.strip()
        if not line:
            continue

        try:
            return json.loads(line)
        except json.JSONDecodeError:
            print("Skipping non-JSON:", line)

In [ ]:
#Handshake
response = send_rpc("initialize", {
    "protocolVersion": "2024-11-05",
    "capabilities": {},
    "clientInfo": {"name": "colab-client", "version": "1.0"}
})

print(response)

In [ ]:
#List the tools
tools = send_rpc("tools/list")
print(tools)
print("------------------------------------")
# Loop and print tool names
for tool in tools["result"]["tools"]:
    print(tool["name"])


In [ ]:
#Test :Call tool
resp = send_rpc("tools/call", {
    "name": "get_answer",
    "arguments": {"question": "Who is Philippe Duval ?",
                  "threshold": 0.5,
                  "k": 2
                  }
})

print(resp)

In [ ]:
# Function to print answer and source
def print_answer_and_source(resp):

  # Step 1: get text payload
  text_payload = resp["result"]["content"][0]["text"]

  # Step 2: convert string → dict
  data = json.loads(text_payload)

  # Step 3: extract fields
  answer = data["answer"]
  sources = data["sources"]

  # Step 4: print nicely
  print("Answer:\n", answer)

  print("\nSources:")
  for i, src in enumerate(sources, 1):
      print(f"{i}. {src[:200]}...")  # truncate for readability

In [ ]:
print_answer_and_source(resp)

In [ ]:
# check that these files exist -> chunks.pkl, faiss.index, mcp_server.py
!ls

# **Let's connect our working MCP server to OpenAI**

In [ ]:
#Step 1 — Define tool schema for OpenAI

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_answer",
            "description": "Answer questions using a knowledge base",
            "parameters": {
                "type": "object",
                "properties": {
                    "question": {
                        "type": "string",
                        "description": "User question"
                    },
                    "threshold": {
                        "type": "number",
                        "description": "Similarity threshold (default 0.45)"
                    },
                    "k": {
                        "type": "integer",
                        "description": "Number of top results to retrieve (default 5)"
                    }
                },
                "required": ["question"]
            }
        }
    }
]


In [ ]:
#Setup OpenAI

from openai import OpenAI
from google.colab import userdata

#define constants
MODEL = "gpt-4.1-mini"

# Sign in to OpenAI using Secrets in Colab
openai_api_key = userdata.get('OPENAI_API_KEY')
client = OpenAI(api_key=openai_api_key)

# Check if Open AI key has been set
if openai_api_key:
    print("OpenAI API Key exists")
else:
    print("OpenAI API Key not set")

# **Let's now create an Agent**

*  Decide when to use tools
*  Decide which tool to use
*  Combine tool output with reasoning

In [ ]:
import json

def agent(question, threshold=0.45, k=5):
    messages = [
        {
            "role": "system",
            "content": "You MUST always use the get_answer tool. Do not answer directly."
        },
        {"role": "user", "content": question}
    ]

    # Force tool usage
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages,
        tools=tools,
        tool_choice={
            "type": "function",
            "function": {"name": "get_answer"}
        }
    )

    msg = response.choices[0].message
    messages.append(msg)

    # Always process tool call (no check needed)
    tool_call = msg.tool_calls[0]

    args = json.loads(tool_call.function.arguments)

    # enforce parameters
    args["threshold"] = threshold
    args["k"] = k

    # call MCP
    mcp_resp = send_rpc("tools/call", {
        "name": "get_answer",
        "arguments": args
    })

    text_payload = mcp_resp["result"]["content"][0]["text"]
    data = json.loads(text_payload)

    # Control point so that answer is given ONLY based on RAG
    if data["answer"] == "DO-NOT-KNOW":
        return "I don't have enough information in the knowledge base to answer this."

    # optional second LLM pass (for nicer wording)
    messages.append({
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": json.dumps(data)
    })

    final = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages
    )

    return final.choices[0].message.content

In [ ]:
print(agent("Who is Philippe Duval married to?"))

In [ ]:
print(agent("Who are the owners of the wine shop in this story ?",0.25))

In [ ]:
print(agent("Who is on trial for treason against England ?"))

In [ ]:
print(agent("Who is the Jackal and who is the Lion ?",0.3,3))

In [ ]:
print(agent("Why is he on trial ?"))

In [ ]:
print(agent("Who is the villain of this story?",0.25))

In [ ]:
print(agent("What is the capital of India ?")) # cannot answer since tool doesn't know

In [ ]:
print(agent("Name the seven wonders of the world ?")) # cannot answer since tool doesn't know